# 07. Transfer to a New Problem

This chapter turns the simple regression example into a reusable workflow.

The goal is not to memorize the fuel consumption example. The goal is to know what to change when the data and problem change.

## A reusable simple regression function

## Browser package setup

Run the next cell before the first analysis cell. It loads the scientific Python packages used in this module into the browser-based Python kernel.

The first run in a browser session can take a little time. Later notebooks usually load faster because the browser can reuse cached packages.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols

from checks import check_close, check_columns, model_snapshot


def fit_slr(data, response, predictor, predict_at=None):
    """Fit a simple linear regression and return model, predictions, and summary values."""
    formula = f"{response} ~ {predictor}"
    model = ols(formula, data=data).fit()
    snapshot = model_snapshot(model)

    prediction = None
    if predict_at is not None:
        new_data = pd.DataFrame({predictor: [predict_at]})
        prediction = model.get_prediction(new_data).summary_frame(alpha=0.05)

    return {
        "formula": formula,
        "model": model,
        "snapshot": snapshot,
        "prediction": prediction,
    }

## Test the function on the original data

In [ ]:
fuel = pd.read_csv("data/fuelcons.csv")

result = fit_slr(
    data=fuel,
    response="Fuelcons",
    predictor="Temp",
    predict_at=40,
)

result["formula"], result["snapshot"], result["prediction"]

In [ ]:
print(check_close(
    "slope from reusable function",
    result["snapshot"]["slope"],
    expected=-0.1279,
    tolerance=0.005,
))

## Transfer data set: campus heating energy

Now use a different small data set. The context is fictional but realistic: weekly campus heating energy use tends to fall as outdoor temperature rises.

In [ ]:
campus = pd.read_csv("data/campus_energy_sample.csv")
campus.head()

In [ ]:
print(check_columns(campus, ["week", "outdoor_temp_f", "heating_mmbtu"]))

## Plot before modeling

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(campus["outdoor_temp_f"], campus["heating_mmbtu"])
ax.set_xlabel("Outdoor temperature (F)")
ax.set_ylabel("Heating energy (MMBtu)")
ax.set_title("Campus heating energy vs outdoor temperature")
plt.show()

## Fit the transferred model

Fill in the response and predictor column names.

In [ ]:
response_col = "heating_mmbtu"
predictor_col = "outdoor_temp_f"
predict_at_value = 60

campus_result = fit_slr(
    data=campus,
    response=response_col,
    predictor=predictor_col,
    predict_at=predict_at_value,
)

campus_result["snapshot"]

In [ ]:
print(check_close(
    "campus slope",
    campus_result["snapshot"]["slope"],
    expected=-2.054,
    tolerance=0.05,
    hint="The response should be heating_mmbtu and the predictor should be outdoor_temp_f."
))
print(check_close(
    "campus predicted mean at 60 F",
    campus_result["prediction"].loc[0, "mean"],
    expected=83.45,
    tolerance=0.5,
))

## Plot the transferred fitted line

In [ ]:
campus_plot = campus.copy()
campus_plot["fitted"] = campus_result["model"].fittedvalues

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(campus_plot[predictor_col], campus_plot[response_col], label="Observed")
ax.plot(campus_plot[predictor_col], campus_plot["fitted"], color="red", label="Fitted line")
ax.set_xlabel("Outdoor temperature (F)")
ax.set_ylabel("Heating energy (MMBtu)")
ax.legend()
plt.show()

## Interpret the transferred model

In [ ]:
campus_snapshot = campus_result["snapshot"]
campus_prediction = campus_result["prediction"]

print(f"Fitted equation: heating_mmbtu = {campus_snapshot['intercept']:.2f} + ({campus_snapshot['slope']:.2f}) * outdoor_temp_f")
print(f"R-squared: {campus_snapshot['r_squared']:.3f}")
print()
print("Prediction at 60 F:")
campus_prediction

```{admonition} Reference interpretation
:class: dropdown

For the campus heating sample, the fitted slope is about -2.05. This means that each additional degree Fahrenheit is associated with about 2.05 fewer MMBtu of expected weekly heating energy use. At 60 F, the fitted mean heating energy is about 83.45 MMBtu. This is a mean estimate, so a prediction interval is needed when forecasting an individual week.
```

## Your own project checklist

When you apply this workflow to homework or a project, write down these answers before modeling:

| Item | Your answer |
|---|---|
| What is one row of the data? | A week, a county, a customer, a product, etc. |
| What is the response variable? | The outcome you want to explain or forecast |
| What is the predictor variable? | The input used to explain the response |
| Is a straight line plausible? | Check the scatter plot |
| Are you estimating a mean or predicting a new observation? | Choose confidence interval or prediction interval |
| What could make the model misleading? | Nonlinearity, omitted variables, time order, outliers, small sample |

## Final checkpoint

Write a conclusion for a data set of your choice. A useful project-style conclusion should include:

- the response and predictor variables,
- the direction and approximate magnitude of the relationship,
- one concrete prediction or fitted value,
- whether the result is a mean estimate or an individual prediction,
- at least one limitation or diagnostic concern.

```{admonition} Reference project-style conclusion
:class: dropdown

In the campus heating sample, outdoor temperature is a strong negative predictor of weekly heating energy use. The fitted slope suggests that each additional degree Fahrenheit is associated with about 2.05 fewer MMBtu of expected weekly heating energy use. At 60 F, the fitted mean is about 83.45 MMBtu. Before using this model for decisions, I would check residual plots, avoid extrapolating outside the observed temperature range, and report a prediction interval for an individual future week.
```

## What to remember

You now have a reusable Python pattern:

1. load the data,
2. identify response and predictor,
3. plot before modeling,
4. fit with `ols("response ~ predictor", data=df).fit()`,
5. interpret slope, R-squared, and intervals,
6. check residual diagnostics,
7. explain the result in the language of the problem.